# AutoDSPy: GPT-2 + GRPO-inspired + GSM8K + Llama 3.1 8B

**All implementation code is in this notebook.** Run cells from top to bottom using
`venv_gpu/Scripts/python.exe` as the VS Code/Jupyter kernel. There are no imports from
the repository's Python modules and no YAML file is required. The previous research
notebook is preserved byte-for-byte as `DSPy_GRPO_Original.ipynb`; its recorded outputs
and existing checkpoints/results remain untouched.

### How to run
1. Start Ollama in a separate terminal (`ollama serve`). The exact `llama3.1:8b`
   model must be installed (`ollama pull llama3.1:8b` if missing).
2. First set **`MODE = "smoke"`** in the configuration cell. Run the setup and
   definition cells, offline tests, endpoint check, dataset loading, and policy setup.
3. Optionally enable the live execution probe. Initialize the training iterator.
   Set **`RUN_ONE_EPISODE = True`** and run its cell, or set
   **`RUN_REMAINING_TRAINING = True`** and run the training loop cell.
4. Inspect the actual logged metrics/checkpoints. Set **`RUN_EVALUATION = True`**
   to evaluate the resulting checkpoint with all four comparison methods.
5. For the full experiment, **restart the kernel**, set **`MODE = "full"`**, run
   setup again, then enable the remaining-training cell. This keeps initialization
   independent of the smoke experiment. Full evaluation uses all 1,319 official test cases.

Full mode is **200 episodes x 200 prompt draws x 5 pipelines = 200,000 pipeline
evaluations**, plus possible grading/adapter calls. Full training is disabled until
you explicitly enable its cell. GPT-2 uses the laptop GPU; Llama defaults to CPU or
a configurable remote Ollama endpoint. Smoke previously took about six minutes;
full CPU execution can take weeks. No published accuracy is claimed here.

### Evidence and unresolved assumptions
- Paper Table 3: GPT-2 (called 127M; actual tied parameters ~124M), Adam 1e-4,
  200 episodes, K=5. Predict/CoT, 15 signatures, first-token action probabilities,
  and module/signature/forced-stop follow the released notebook.
- Figure 2 and released code use `-sum(log_probability * (reward - group_mean))`,
  **without clipping, standard-deviation normalization, entropy, GAE, or KL**.
  Shared-table gamma=.99, lambda=.95, epsilon=.2 and entropy=.01 are recorded but
  inactive in the default objective. Optional sensitivity objectives are visible below.
- 200 independent train draws with replacement **per episode**, one Adam step per
  episode, seed 42, zero policy dropout, pinned revisions, generation settings,
  and numerical reward repairs are explicitly documented assumptions/repairs.
  The main-text 200-example pool conflicts with appendix per-episode wording.
- Continuous LLM fallback rewards follow released code; evaluation reports numeric
  exact match and binary LLM fallback separately. The appendix's unspecified embedding
  model is not invented. Official test has 1,319 cases; paper says 1,300 with unknown indices.
- Llama quantization/runtime are measured, not presumed equal to the paper's runtime.

Paper: `AutoDSPy.pdf` (Table 3, Figure 2, A.4, A.7).
Released reference: https://github.com/nafew-azim/AUTODSPy/blob/ead26600a23febd14f28f5b0a66966eaff7fbbf8/DSPy_GRPO.ipynb
Detailed evidence remains in `docs/REPRODUCTION_AUDIT.md` and `docs/DEVIATIONS_FROM_PAPER.md`.


## 1. Environment and notebook provenance

Select the existing GPU environment as the notebook kernel. Dependencies are PyTorch,
Transformers, DSPy, datasets, NumPy, and PyYAML (the latter is only needed by the
optional YAML helper). No install command runs automatically and no results are deleted.
If dependencies are missing, install `requirements-reproduction.txt` into the selected
environment; install CUDA PyTorch separately from its cu124 index. `ipykernel` is
already installed in this checkout's GPU environment.

Source hashes below record the **actually executed definition cells**, independently
of saved notebook outputs and run-control settings. This permits episode-boundary
resume after a kernel restart, while rejecting changes to the implementation.


In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # Before CUDA initialization.
import sys
import json
import hashlib
from pathlib import Path
from dataclasses import replace

NOTEBOOK_NAME = "DSPy_GRPO.ipynb"
ROOT = Path.cwd()
assert (ROOT / NOTEBOOK_NAME).is_file(), "Open the notebook with the repository root as working directory."
NOTEBOOK_IMPLEMENTATION_SOURCES = {}

def register_implementation(cell_id):
    # In Jupyter/IPython capture the code just executed, including unsaved edits.
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    if shell is not None:
        source = shell.history_manager.input_hist_raw[-1]
    else:
        notebook = json.loads((ROOT / NOTEBOOK_NAME).read_text(encoding="utf-8"))
        cell = next(cell for cell in notebook["cells"] if cell["id"] == cell_id)
        source = "".join(cell["source"])
    NOTEBOOK_IMPLEMENTATION_SOURCES[cell_id] = source

print("Kernel:", sys.executable)
print("Working directory:", ROOT)


Kernel: x:\Adaptive GRPO for AutoDSPy\Code\AUTODSPy\venv_gpu\Scripts\python.exe
Working directory: x:\Adaptive GRPO for AutoDSPy\Code\AUTODSPy


## 2. Complete configuration definition


In [2]:
"""Strict configuration; paper settings and active algorithm settings are separate."""
from dataclasses import asdict, dataclass, fields
from pathlib import Path
import math
import yaml


@dataclass(frozen=True)
class Config:
    seed: int = 42
    policy_model: str = "gpt2"
    policy_revision: str = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
    dataset_revision: str = "740312add88f781978c0658806c59bc2815b9866"
    device: str = "cuda"
    episodes: int = 200
    samples_per_episode: int = 200
    group_size: int = 5
    learning_rate: float = 1e-4
    objective: str = "notebook"
    advantage_mode: str = "centered"
    clip_epsilon: float = 0.2
    entropy_coefficient: float = 0.0
    paper_gamma: float = 0.99
    paper_lambda: float = 0.95
    paper_entropy_coefficient: float = 0.01
    train_pool_size: int | None = None
    max_policy_tokens: int = 1024
    gradient_checkpointing: bool = False
    cpu_threads: int = 4
    checkpoint_every: int = 10
    execution_model: str = "llama3.1:8b"
    execution_endpoint: str = "http://localhost:11434"
    execution_num_gpu: int = 0
    execution_temperature: float = 0.0
    execution_max_tokens: int = 4096
    execution_context: int = 8192
    execution_timeout: int = 300
    execution_keep_alive: str = "5m"
    semantic_fallback: bool = True
    evaluation_limit: int | None = None
    output_root: str = "runs/reproduction"

    def validate(self):
        if self.policy_model != "gpt2" or self.execution_model != "llama3.1:8b":
            raise ValueError("This reproduction requires GPT-2 and llama3.1:8b; no substitution.")
        for name in ("episodes", "samples_per_episode", "cpu_threads", "checkpoint_every", "max_policy_tokens",
                     "execution_max_tokens", "execution_context", "execution_timeout"):
            if type(getattr(self, name)) is not int or getattr(self, name) <= 0:
                raise ValueError(f"{name} must be a positive integer")
        if type(self.group_size) is not int or self.group_size < 2:
            raise ValueError("group_size must be >= 2")
        if not 1 <= self.max_policy_tokens <= 1024:
            raise ValueError("GPT-2 supports at most 1024 policy tokens")
        if self.objective not in ("notebook", "clipped"):
            raise ValueError("objective must be notebook or clipped")
        if self.advantage_mode not in ("centered", "standardized"):
            raise ValueError("advantage_mode must be centered or standardized")
        if self.objective == "notebook" and self.entropy_coefficient != 0:
            raise ValueError("Released notebook objective has no entropy regularization")
        if self.device not in ("cpu", "cuda"):
            raise ValueError("device must be cpu or cuda")
        if not math.isfinite(self.learning_rate) or self.learning_rate <= 0:
            raise ValueError("learning_rate must be finite and positive")
        if not 0 < self.clip_epsilon < 1 or not math.isfinite(self.entropy_coefficient) or self.entropy_coefficient < 0:
            raise ValueError("Invalid clipping or entropy settings")
        if self.execution_num_gpu < 0:
            raise ValueError("Specify CPU (0) or an explicit positive Ollama GPU-layer count")
        if not math.isfinite(self.execution_temperature) or self.execution_temperature < 0:
            raise ValueError("Invalid execution temperature")
        for name in ("train_pool_size", "evaluation_limit"):
            value = getattr(self, name)
            if value is not None and (type(value) is not int or value <= 0):
                raise ValueError(f"{name} must be null or a positive integer")
        if not self.execution_endpoint.startswith(("http://", "https://")):
            raise ValueError("execution_endpoint must be an HTTP(S) Ollama URL")
        return self

    def to_dict(self):
        return asdict(self)


def load_config(filename):
    data = yaml.safe_load(Path(filename).read_text(encoding="utf-8-sig"))
    if not isinstance(data, dict):
        raise ValueError("Configuration must be a YAML mapping")
    unknown = set(data) - {f.name for f in fields(Config)}
    if unknown:
        raise ValueError(f"Unknown configuration fields: {sorted(unknown)}")
    return Config(**data).validate()

register_implementation("config-definition")


### Select full or smoke; configure execution placement here

All values are visible below. No YAML is loaded. Change `MODE` before setup.
Keep full budget, K, and model identifiers unchanged for the target experiment.
`execution_num_gpu=0` means Llama CPU execution; for a remote GPU server use its
endpoint and an explicit positive GPU-layer count. Use identical execution settings
for training and evaluation. CPU execution does not reduce the model size.


In [3]:
MODE = "full"  # First milestone: change to "smoke". Restart kernel before a new independent run.

FULL_CONFIG = Config(
    seed=42,
    policy_model="gpt2",
    policy_revision="607a30d783dfa663caf39e06633721c8d4cfcd7e",
    dataset_revision="740312add88f781978c0658806c59bc2815b9866",
    device="cuda",
    episodes=200,
    samples_per_episode=200,
    group_size=5,
    learning_rate=1e-4,
    objective="notebook",
    advantage_mode="centered",
    entropy_coefficient=0.0,  # Released GRPO has no entropy term.
    clip_epsilon=0.2,        # Inactive in default notebook objective.
    paper_gamma=0.99,        # Evidence metadata; no GAE in released GRPO.
    paper_lambda=0.95,
    paper_entropy_coefficient=0.01,
    train_pool_size=None,    # Full train pool; 200 new draws per episode, with replacement.
    max_policy_tokens=1024,
    gradient_checkpointing=False,
    cpu_threads=4,
    checkpoint_every=10,
    execution_model="llama3.1:8b",
    execution_endpoint="http://localhost:11434",  # Or http://REMOTE_HOST:11434
    execution_num_gpu=0,
    execution_temperature=0.0,
    execution_max_tokens=4096,
    execution_context=8192,
    execution_timeout=300,
    execution_keep_alive="5m",
    semantic_fallback=True,
    evaluation_limit=None,  # Full official held-out test: 1,319 cases.
    output_root="runs/reproduction-notebook",
).validate()
SMOKE_CONFIG = replace(FULL_CONFIG, episodes=1, samples_per_episode=2,
                       checkpoint_every=1, evaluation_limit=2,
                       output_root="runs/reproduction-notebook-smoke").validate()
if MODE not in ("full", "smoke"):
    raise ValueError('MODE must be "full" or "smoke"')
config = FULL_CONFIG if MODE == "full" else SMOKE_CONFIG
print(json.dumps(config.to_dict(), indent=2))
print("Candidate pipeline evaluations:", config.episodes * config.samples_per_episode * config.group_size)


{
  "seed": 42,
  "policy_model": "gpt2",
  "policy_revision": "607a30d783dfa663caf39e06633721c8d4cfcd7e",
  "dataset_revision": "740312add88f781978c0658806c59bc2815b9866",
  "device": "cuda",
  "episodes": 200,
  "samples_per_episode": 200,
  "group_size": 5,
  "learning_rate": 0.0001,
  "objective": "notebook",
  "advantage_mode": "centered",
  "clip_epsilon": 0.2,
  "entropy_coefficient": 0.0,
  "paper_gamma": 0.99,
  "paper_lambda": 0.95,
  "paper_entropy_coefficient": 0.01,
  "train_pool_size": null,
  "max_policy_tokens": 1024,
  "gradient_checkpointing": false,
  "cpu_threads": 4,
  "checkpoint_every": 10,
  "execution_model": "llama3.1:8b",
  "execution_endpoint": "http://localhost:11434",
  "execution_num_gpu": 0,
  "execution_temperature": 0.0,
  "execution_max_tokens": 4096,
  "execution_context": 8192,
  "execution_timeout": 300,
  "execution_keep_alive": "5m",
  "semantic_fallback": true,
  "evaluation_limit": null,
  "output_root": "runs/reproduction-notebook"
}
Candidate

## 3. Canonical numerical answers and LLM reward

Raw GSM8K reasoning is retained. Exact match handles signed numbers, commas and decimals; failed exact match uses the released continuous LLM reward during training and a binary fallback during evaluation. Infrastructure failures propagate.


In [4]:
"""Canonical GSM8K numeric exact match and released LLM reward fallback."""
from dataclasses import dataclass, asdict
from decimal import Decimal, InvalidOperation
import re
import math

NUMBER = re.compile(r"[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|[+-]?\.\d+")
BRACKET = re.compile(r"\[([^\[\]]*)\]")


def canonical_answer(raw_answer):
    if "####" not in raw_answer:
        raise ValueError("Official GSM8K answer is missing the #### delimiter")
    return raw_answer.rsplit("####", 1)[1].strip()


def normalize_number(text):
    if text is None:
        return None
    text = str(text).replace("\u2212", "-").strip()
    if "####" in text:
        text = text.rsplit("####", 1)[1].strip()
    brackets = BRACKET.findall(text)
    if brackets:
        matches = NUMBER.findall(brackets[-1])
        # Reject ambiguous bracketed answers such as [3 or 4].
        if len(matches) != 1:
            return None
        number = matches[0]
    else:
        matches = NUMBER.findall(text)
        if not matches:
            return None
        number = matches[-1]
    try:
        value = Decimal(number.replace(",", ""))
        return value if value.is_finite() else None
    except InvalidOperation:
        return None


def exact_match(response, answer):
    left, right = normalize_number(response), normalize_number(answer)
    return left is not None and right is not None and left == right


def parse_judge_score(text, binary=False):
    brackets = BRACKET.findall(str(text))
    candidate = brackets[-1].strip() if brackets else str(text).strip()
    if not re.fullmatch(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)", candidate):
        raise ValueError(f"Unparseable judge score: {text!r}")
    value = float(candidate)
    if not math.isfinite(value) or (binary and value not in (0.0, 1.0)):
        raise ValueError(f"Invalid judge score: {text!r}")
    return min(max(value, 0.0), 1.0)


@dataclass(frozen=True)
class Reward:
    value: float
    exact: bool
    fallback_used: bool = False
    judge_text: str | None = None
    error: str | None = None

    def to_dict(self):
        return asdict(self)


def compute_reward(prompt, response, answer, judge=None, binary=False):
    if response is None:
        return Reward(0.0, False)
    if exact_match(response, answer):
        return Reward(1.0, True)
    if judge is None:
        return Reward(0.0, False)
    if binary:
        instruction = ("Evaluate whether the following response correctly answers the prompt based on the ground truth.\n"
                       "Return a score of 1.0 or 0.0 if the response is correct and 0 if incorrect.\n"
                       "Only return the score inside the square brackets [].")
    else:
        instruction = ("Evaluate whether the following response correctly answers the prompt based on the ground truth "
                       "and give your final score in the square brackets[final score]. only the value in the [] "
                       "should in your response and nothing else.")
    query = f"{instruction}\nPrompt: {prompt}\nResponse: {response}\nGround Truth: {answer}"
    if not binary:
        query += ("\nReturn a score in range between 1.0 to 0.0 if the response is correct or partially correct "
                  "(matches or is equivalent to the ground truth), or 0.0 if incorrect.\nfinal score:[]")
    else:
        query += "\nFinal score:[]"
    text = judge(query)  # Infrastructure errors must abort rather than train on zero rewards.
    try:
        return Reward(parse_judge_score(text, binary), False, True, str(text))
    except ValueError as exc:
        return Reward(0.0, False, True, str(text), str(exc))

register_implementation("reward-definition")


## 4. Official GSM8K loading and split guards

Training loads only the official train split. Held-out test loading occurs later, after training. No HotpotQA is loaded.


In [5]:
"""Official splits; training never loads the held-out split."""
from dataclasses import dataclass, asdict
import hashlib
import json
import random


@dataclass(frozen=True)
class Example:
    index: int
    question: str
    raw_answer: str
    answer: str
    split: str

    def to_dict(self):
        return asdict(self)


def load_examples(config, split):
    from datasets import load_dataset
    if split not in ("train", "test"):
        raise ValueError("Only official train/test splits are allowed")
    dataset = load_dataset("openai/gsm8k", "main", split=split, revision=config.dataset_revision)
    rows = [Example(i, row["question"], row["answer"], canonical_answer(row["answer"]), split)
            for i, row in enumerate(dataset)]
    manifest = {"dataset": "openai/gsm8k", "config": "main", "split": split,
                "revision": config.dataset_revision, "fingerprint": dataset._fingerprint,
                "official_rows": len(rows)}
    if split == "train" and config.train_pool_size is not None:
        if config.train_pool_size > len(rows):
            raise ValueError("Training pool exceeds official split")
        rows = random.Random(config.seed).sample(rows, config.train_pool_size)
    if split == "test" and config.evaluation_limit is not None:
        if config.evaluation_limit > len(rows):
            raise ValueError("Evaluation limit exceeds official split")
        rows = random.Random(config.seed).sample(rows, config.evaluation_limit)
    manifest["selected_indices"] = [row.index for row in rows]
    manifest["content_sha256"] = hashlib.sha256(json.dumps(
        [row.to_dict() for row in rows], sort_keys=True).encode()).hexdigest()
    return rows, manifest


def check_no_leakage(train, test):
    if any(row.split != "train" for row in train) or any(row.split != "test" for row in test):
        raise ValueError("Incorrect train/test split labels")
    overlap = {r.question.strip() for r in train} & {r.question.strip() for r in test}
    if overlap:
        raise ValueError(f"Found {len(overlap)} overlapping train/test questions")

register_implementation("dataset-definition")


## 5. GPT-2 policy, full action space, sampling and differentiable replay

Duplicate first-token IDs stay as distinct action slots. This preserves the released action-selection limitation. Each pipeline samples a module, a signature, then deterministic stop. No ReAct, retrieval, tools or teleprompter.


In [6]:
"""Released first-token action distribution, including repeated token IDs."""
from dataclasses import dataclass, asdict
import torch
from torch.distributions import Categorical

MODULES = ("CoT", "Predict")
SIGNATURES = (
    "question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
    "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
    "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
    "text -> response", "prompt -> generated_text", "query -> generated_text",
)


@dataclass(frozen=True)
class Decision:
    state: str
    valid_actions: tuple[str, ...]
    choice: int
    old_log_prob: float


@dataclass(frozen=True)
class Trajectory:
    pipeline: tuple[str, ...]
    decisions: tuple[Decision, ...]

    def to_dict(self):
        return asdict(self)


def action_distribution(vocab_logits, tokenizer, actions):
    ids = [tokenizer.encode(a, add_special_tokens=False) for a in actions]
    if not ids or any(not tokens for tokens in ids):
        raise ValueError("Every action must tokenize to at least one token")
    # Repeated IDs remain repeated slots. Deduplicating changes the experiment.
    logits = vocab_logits[[tokens[0] for tokens in ids]].float()
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Non-finite action logits")
    return Categorical(logits=logits)


class Policy:
    def __init__(self, model, tokenizer, max_tokens=1024, gradient_checkpointing=False):
        self.model, self.tokenizer = model, tokenizer
        self.max_tokens = max_tokens
        self.gradient_checkpointing = gradient_checkpointing
        # Explicit replay-consistency assumption; train/eval preserve gradients.
        for module in model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.p = 0.0
        if gradient_checkpointing:
            model.gradient_checkpointing_enable()
        model.eval()

    @property
    def device(self):
        return next(self.model.parameters()).device

    def distribution(self, state, actions):
        if actions == ("stop",):
            # Deterministic termination has probability 1 and needs no GPT-2 call.
            return Categorical(logits=torch.zeros(1, device=self.device))
        inputs = self.tokenizer(state, return_tensors="pt", padding=True,
                                truncation=True, max_length=self.max_tokens).to(self.device)
        # Exactly the final-position GPT-2 logits; avoid allocating [T, vocab] logits.
        if hasattr(self.model, "transformer") and hasattr(self.model, "lm_head"):
            hidden = self.model.transformer(**inputs, use_cache=False).last_hidden_state
            logits = self.model.lm_head(hidden[:, -1, :])[0]
        else:
            logits = self.model(**inputs).logits[0, -1]
        return action_distribution(logits, self.tokenizer, actions)

    @torch.no_grad()
    def generate(self, prompt, greedy=False):
        self.model.eval()
        pipeline, decisions = [], []
        for actions in (MODULES, SIGNATURES, ("stop",)):
            state = f"Prompt: {prompt} Pipeline: {' '.join(pipeline)}"
            dist = self.distribution(state, actions)
            choice = int(dist.probs.argmax()) if greedy else int(dist.sample())
            decisions.append(Decision(state, actions, choice, float(dist.log_prob(
                torch.tensor(choice, device=self.device)))))
            pipeline.append(actions[choice])
        return Trajectory(tuple(pipeline), tuple(decisions))

    def score(self, decision):
        self.model.train(self.gradient_checkpointing)
        dist = self.distribution(decision.state, decision.valid_actions)
        return dist.log_prob(torch.tensor(decision.choice, device=self.device)), dist.entropy()

    def token_collisions(self):
        result = {}
        for action in MODULES + SIGNATURES + ("stop",):
            token = self.tokenizer.encode(action, add_special_tokens=False)[0]
            result.setdefault(str(token), []).append(action)
        return {token: actions for token, actions in result.items() if len(actions) > 1}


def load_policy(config):
    from transformers import GPT2Tokenizer, GPT2LMHeadModel
    if config.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable; explicitly select device=cpu")
    tokenizer = GPT2Tokenizer.from_pretrained(config.policy_model, revision=config.policy_revision)
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(config.policy_model, revision=config.policy_revision)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.to(config.device)
    return Policy(model, tokenizer, config.max_policy_tokens, config.gradient_checkpointing)

register_implementation("policy-definition")


## 6. Ollama identity checks and sequential DSPy execution

The endpoint check requires the exact Llama 3.1 8B tag, records digest/quantization/settings, and fails instead of substituting a smaller model. All four evaluation methods use the same DSPy execution instructions.


In [7]:
"""Sequential DSPy execution through a validated, configurable Ollama server."""
from dataclasses import dataclass, asdict
import json
import time
import urllib.request


def ollama_json(endpoint, route, payload=None, timeout=10):
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(endpoint.rstrip("/") + route, data=data,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)


def preflight(config):
    version = ollama_json(config.execution_endpoint, "/api/version")
    models = ollama_json(config.execution_endpoint, "/api/tags")["models"]
    found = next((m for m in models if m.get("name") == config.execution_model
                  or m.get("model") == config.execution_model), None)
    if found is None:
        raise RuntimeError(f"Required {config.execution_model} missing at {config.execution_endpoint}. "
                           "Install it on that server; smaller models are not substitutes.")
    info = ollama_json(config.execution_endpoint, "/api/show", {"model": config.execution_model})
    details = info.get("details", {})
    if details.get("family") != "llama" or not str(details.get("parameter_size", "")).startswith("8"):
        raise RuntimeError("Ollama tag does not identify an 8B Llama model")
    if not details.get("quantization_level"):
        raise RuntimeError("Cannot record execution-model quantization")
    model_info = info.get("model_info", {})
    if "Llama-3.1" not in str(model_info.get("general.basename", "")):
        raise RuntimeError("Cannot verify that this tag contains Llama 3.1")
    # License text and tensor listings are large and do not identify runtime behavior.
    recorded_info = {key: info.get(key) for key in ("details", "model_info", "parameters", "template", "capabilities")}
    return {"version": version, "model": found, "show": recorded_info,
            "endpoint": config.execution_endpoint, "settings": {
                "num_gpu": config.execution_num_gpu, "temperature": config.execution_temperature,
                "num_predict": config.execution_max_tokens, "num_ctx": config.execution_context,
                "seed": config.seed, "keep_alive": config.execution_keep_alive,
                "timeout": config.execution_timeout, "cache": False}}


@dataclass(frozen=True)
class Execution:
    response: str | None
    seconds: float
    error: str | None = None

    def to_dict(self):
        return asdict(self)


def pipeline_pair(pipeline):
    pipeline = tuple(pipeline)
    if pipeline and pipeline[-1] == "stop":
        pipeline = pipeline[:-1]
    if len(pipeline) != 2 or pipeline[0] not in MODULES or pipeline[1] not in SIGNATURES:
        return None
    return pipeline


class DSPyExecutor:
    def __init__(self, config):
        import dspy
        self.config = config
        self.lm = dspy.LM("ollama_chat/" + config.execution_model,
                          api_base=config.execution_endpoint, api_key="",
                          temperature=config.execution_temperature, max_tokens=config.execution_max_tokens,
                          num_ctx=config.execution_context, num_gpu=config.execution_num_gpu,
                          seed=config.seed, keep_alive=config.execution_keep_alive,
                          timeout=config.execution_timeout, cache=False, num_retries=0)

    def runtime(self):
        return ollama_json(self.config.execution_endpoint, "/api/ps")

    def execute(self, prompt, pipeline):
        import dspy
        pair = pipeline_pair(pipeline)
        if pair is None:
            return Execution(None, 0.0, "invalid_pipeline")
        module, signature = pair
        input_field, output_field = (field.strip() for field in signature.split("->"))
        program = dspy.ChainOfThought(signature) if module == "CoT" else dspy.Predict(signature)
        formatted = ("system instruction: Must give your final answer in square brackets without fail! "
                     f"e.g., [final answer] like this: [5] \n prompt: {prompt}")
        started = time.perf_counter()
        # Same executor/settings/instructions are used for learned and static methods.
        from dspy.utils.exceptions import AdapterParseError
        try:
            with dspy.context(lm=self.lm, disable_history=True):
                result = program(**{input_field: formatted})
        except AdapterParseError as exc:
            return Execution(None, time.perf_counter() - started, f"adapter_parse_error: {exc}")
        value = result.get(output_field)
        return Execution(str(value) if value is not None else None,
                         time.perf_counter() - started, None if value is not None else "missing_output_field")

    def judge(self, prompt):
        import dspy
        with dspy.context(disable_history=True):
            return self.lm(prompt)[0]

register_implementation("execution-definition")


## 7. Reproducible seeds, logs, memory measurements and checkpoints

Unique run directories preserve prior results. Checkpoints include policy, Adam and RNG states. Resume restores completed episode boundaries; unsaved interrupted episodes must be repeated. Policy peak CUDA memory excludes other processes and the execution server.


In [8]:
"""Run isolation, provenance, episode-boundary checkpoints and deterministic RNG."""
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import subprocess
import uuid
import numpy as np
import torch


def seed_everything(seed, cpu_threads=4):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(cpu_threads)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # On Windows CUDA set CUBLAS_WORKSPACE_CONFIG before starting Python.
    torch.use_deterministic_algorithms(True)


def json_write(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


def append_json(path, value):
    with Path(path).open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(value, allow_nan=False) + "\n")


def new_run(output_root, label="train"):
    parent = Path(output_root)
    parent.mkdir(parents=True, exist_ok=True)
    run = parent / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + label + "-" + uuid.uuid4().hex[:8])
    run.mkdir()  # Never reuse or clear an existing run.
    return run


def shell_output(args):
    try:
        return subprocess.check_output(args, text=True, stderr=subprocess.STDOUT, timeout=15).strip()
    except (OSError, subprocess.SubprocessError):
        return None


def environment_manifest():
    versions = {}
    for package in ("torch", "transformers", "dspy", "datasets", "numpy", "PyYAML", "litellm", "huggingface-hub"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    source_hashes = {f"notebook-cell:{cell_id}": hashlib.sha256(source.encode("utf-8")).hexdigest()
                     for cell_id, source in sorted(NOTEBOOK_IMPLEMENTATION_SOURCES.items())}
    return {"python": platform.python_version(), "platform": platform.platform(), "versions": versions,
            "source_sha256": source_hashes, "entrypoint": NOTEBOOK_NAME,
            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
            "git_head": shell_output(["git", "rev-parse", "HEAD"]),
            "git_status": shell_output(["git", "status", "--short"]),
            "cuda_runtime": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "nvidia_smi": shell_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version",
                                        "--format=csv,noheader"])}


def cuda_memory(device):
    if str(device).startswith("cuda"):
        torch.cuda.synchronize()
        return {"policy_peak_allocated_bytes": torch.cuda.max_memory_allocated(),
                "policy_peak_reserved_bytes": torch.cuda.max_memory_reserved()}
    return {"policy_peak_allocated_bytes": None, "policy_peak_reserved_bytes": None}


def artifact_hash(filename):
    digest = hashlib.sha256()
    with Path(filename).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_checkpoint(filename, policy, optimizer, config, episode, sampler, provenance):
    filename = Path(filename)
    if filename.exists():
        raise FileExistsError(f"Refusing to overwrite checkpoint: {filename}")
    payload = {"format_version": 1, "model": policy.model.state_dict(),
               "optimizer": optimizer.state_dict() if optimizer is not None else None,
               "config": config.to_dict(), "completed_episodes": episode,
               "torch_rng": torch.get_rng_state(), "python_rng": random.getstate(),
               "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
               "sampler_rng": sampler.bit_generator.state, "provenance": provenance}
    temporary = filename.with_suffix(".partial")
    torch.save(payload, temporary)
    temporary.replace(filename)


def load_checkpoint(filename, policy, optimizer=None, sampler=None, expected_config=None, restore_rng=False):
    payload = torch.load(filename, map_location="cpu", weights_only=True)
    if payload.get("format_version") != 1:
        raise ValueError("Not an isolated reproduction checkpoint; legacy weights require a separate conversion")
    if expected_config is not None and payload["config"] != expected_config.to_dict():
        raise ValueError("Resume config differs from checkpoint; resume requires identical settings")
    policy.model.load_state_dict(payload["model"])
    if optimizer is not None:
        if payload["optimizer"] is None:
            raise ValueError("Initial-policy checkpoint has no optimizer state")
        optimizer.load_state_dict(payload["optimizer"])
    if restore_rng:
        if sampler is None:
            raise ValueError("A sampler is required when restoring RNG")
        sampler.bit_generator.state = payload["sampler_rng"]
        torch.set_rng_state(payload["torch_rng"])
        random.setstate(payload["python_rng"])
        if payload["cuda_rng"]:
            if not torch.cuda.is_available():
                raise ValueError("CUDA RNG cannot be resumed without CUDA")
            torch.cuda.set_rng_state_all(payload["cuda_rng"])
    return payload

register_implementation("runtime-definition")


## 8. Group advantages and the actual released objective

Default: centered rewards, sum over candidate/action losses, average over prompt groups. Equal-reward groups contribute zero gradient. The optional clipped/standardized functions are sensitivity controls, not reproduction defaults.


In [9]:
"""Actual released objective and explicitly labeled clipped sensitivity objective."""
import torch


def group_advantages(rewards, mode="centered"):
    values = torch.as_tensor(rewards, dtype=torch.float32).detach()
    if values.ndim != 1 or values.numel() < 2 or not torch.isfinite(values).all():
        raise ValueError("A group requires at least two finite rewards")
    if mode not in ("centered", "standardized"):
        raise ValueError("Unknown advantage mode")
    if torch.equal(values, values[0].expand_as(values)):
        return torch.zeros_like(values)
    advantages = values - values.mean()
    if mode == "standardized":
        advantages = advantages / values.std(correction=0).clamp_min(1e-8)
    return advantages


def action_loss(current_log_prob, old_log_prob, advantage, entropy,
                objective="notebook", clip_epsilon=0.2, entropy_coefficient=0.0):
    old = torch.as_tensor(old_log_prob, device=current_log_prob.device).detach()
    adv = torch.as_tensor(advantage, device=current_log_prob.device).detach()
    if objective == "notebook":
        if entropy_coefficient != 0:
            raise ValueError("Notebook objective has no entropy term")
        return -current_log_prob * adv
    if objective != "clipped":
        raise ValueError("Unknown objective")
    ratio = torch.exp(current_log_prob - old)
    surrogate = torch.minimum(ratio * adv, ratio.clamp(1-clip_epsilon, 1+clip_epsilon) * adv)
    return -surrogate - entropy_coefficient * entropy


def group_loss(current, old, rewards, entropies, objective="notebook",
               advantage_mode="centered", clip_epsilon=0.2, entropy_coefficient=0.0):
    """[K,T] arrays; sums actions/group exactly as the released notebook."""
    if current.ndim != 2 or current.shape != old.shape or current.shape != entropies.shape:
        raise ValueError("Log probabilities and entropies must have identical [K,T] shapes")
    advantages = group_advantages(rewards, advantage_mode).to(current.device)
    if advantages.numel() != current.shape[0]:
        raise ValueError("Reward count must match group size")
    return action_loss(current, old, advantages[:, None], entropies,
                       objective, clip_epsilon, entropy_coefficient).sum()

register_implementation("loss-definition")


## 9. Gradient accumulation and one Adam update per episode

Collect all groups at unchanged policy weights, then replay one action at a time. The behavior/current probability check detects inconsistent replay before an update. This reduces activation memory without changing the accumulated objective.


In [10]:
"""Bounded activation memory: episode rollouts then sequential action backward."""
from pathlib import Path
import time
import numpy as np
import torch


def update_episode(policy, optimizer, groups, config):
    if not groups:
        raise ValueError("An episode must contain prompt groups")
    optimizer.zero_grad(set_to_none=True)
    total_loss, max_log_difference = 0.0, 0.0
    for trajectories, rewards in groups:
        advantages = group_advantages(rewards, config.advantage_mode)
        for trajectory, advantage in zip(trajectories, advantages):
            for decision in trajectory.decisions:
                if decision.valid_actions == ("stop",):
                    continue
                current, entropy = policy.score(decision)
                difference = abs(float(current.detach()) - decision.old_log_prob)
                max_log_difference = max(max_log_difference, difference)
                # No optimizer step until ALL episode actions have been replayed.
                if difference > 2e-4:
                    raise RuntimeError("Behavior/current probabilities differ before update; replay is inconsistent")
                loss = action_loss(current, decision.old_log_prob, advantage, entropy,
                                   config.objective, config.clip_epsilon, config.entropy_coefficient) / len(groups)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Non-finite loss")
                total_loss += float(loss.detach())
                loss.backward()
    squared_norm = torch.zeros((), device=policy.device)
    for parameter in policy.model.parameters():
        if parameter.grad is not None:
            if not torch.isfinite(parameter.grad).all():
                raise FloatingPointError("Non-finite policy gradient; optimizer not stepped")
            squared_norm += parameter.grad.detach().float().square().sum()
    grad_norm = float(squared_norm.sqrt())
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    return {"loss": total_loss, "gradient_norm": grad_norm,
            "max_behavior_current_logprob_difference": max_log_difference}

register_implementation("update-definition")


## 10. Complete episode loop, logging and checkpoint logic

The iterator below is the full training implementation. It yields after each completed
episode and checkpoint decision, allowing cell-by-cell inspection without changing
sampling, optimizer cadence or RNG. Exhaust the iterator to finalize `status.json`.
The one-shot `train` wrapper below exists for offline parity tests; interactive controls
use this same iterator. The first advance saves initial weights and allocates Adam.


In [11]:
def train_episodes(config, policy, examples, executor, provenance, resume=None):
    config.validate()
    if not examples or any(example.split != "train" for example in examples):
        raise ValueError("Training requires nonempty official train examples only")
    run = new_run(config.output_root)
    json_write(run / "config.json", config.to_dict())
    provenance = dict(provenance, algorithm={"optimizer": "Adam", "betas": [0.9, 0.999],
        "epsilon": 1e-8, "weight_decay": 0.0, "optimizer_steps_per_episode": 1,
        "reduction": "sum_over_candidates_and_actions_mean_over_prompt_groups",
        "advantage_mode": config.advantage_mode, "objective": config.objective,
        "GAE": False, "KL_penalty": False})
    json_write(run / "manifest.json", provenance)
    optimizer = torch.optim.Adam(policy.model.parameters(), lr=config.learning_rate)
    sampler = np.random.default_rng(config.seed)
    first_episode = 0
    if resume is not None:
        payload = load_checkpoint(resume, policy, optimizer, sampler, config, restore_rng=True)
        previous = payload["provenance"]
        for key in ("data", "policy", "execution"):
            if previous.get(key) != provenance.get(key):
                raise ValueError(f"Resume provenance changed: {key}")
        if previous.get("environment", {}).get("source_sha256") != provenance.get("environment", {}).get("source_sha256"):
            raise ValueError("Resume implementation source changed")
        first_episode = payload["completed_episodes"]
        provenance = dict(provenance, initial_checkpoint=previous["initial_checkpoint"],
                          initial_checkpoint_sha256=previous.get("initial_checkpoint_sha256"),
                          resumed_from=str(Path(resume).resolve()))
    else:
        initial = run / "initial_policy.pt"
        # Store identity in every checkpoint so evaluation uses the correct untrained model.
        provenance = dict(provenance, initial_checkpoint=str(initial.resolve()))
        save_checkpoint(initial, policy, None, config, 0, sampler, provenance)
        provenance["initial_checkpoint_sha256"] = artifact_hash(initial)
    json_write(run / "manifest.json", provenance)
    if first_episode >= config.episodes:
        raise ValueError("Checkpoint already completed configured episode budget")
    if policy.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    try:
        for episode in range(first_episode, config.episodes):
            started = time.perf_counter()
            positions = sampler.integers(0, len(examples), size=config.samples_per_episode)
            groups, rewards_flat, execution_times, generation_times = [], [], [], []
            zero_groups = 0
            for sample, position in enumerate(positions):
                example = examples[int(position)]
                trajectories, rewards = [], []
                for candidate in range(config.group_size):
                    generated = time.perf_counter()
                    trajectory = policy.generate(example.question)
                    generation_seconds = time.perf_counter() - generated
                    execution = executor.execute(example.question, trajectory.pipeline)
                    judged = time.perf_counter()
                    reward = compute_reward(example.question, execution.response, example.answer,
                                            executor.judge if config.semantic_fallback else None)
                    judge_seconds = time.perf_counter() - judged
                    append_json(run / "trajectories.jsonl", {
                        "episode": episode + 1, "sample": sample, "candidate": candidate,
                        "example": example.to_dict(), "trajectory": trajectory.to_dict(),
                        "execution": execution.to_dict(), "reward": reward.to_dict(),
                        "generation_seconds": generation_seconds, "judge_seconds": judge_seconds})
                    trajectories.append(trajectory)
                    rewards.append(reward.value)
                    rewards_flat.append(reward.value)
                    execution_times.append(execution.seconds)
                    generation_times.append(generation_seconds)
                zero_groups += int(len(set(rewards)) == 1)
                groups.append((trajectories, rewards))
                print(f"episode={episode+1} sample={sample+1}/{config.samples_per_episode} "
                      f"train_index={example.index} rewards={rewards}", flush=True)
            update = update_episode(policy, optimizer, groups, config)
            metrics = {"episode": episode + 1, "samples": len(groups),
                       "pipeline_evaluations": len(rewards_flat),
                       "train_indices": [examples[int(i)].index for i in positions],
                       "unique_train_indices": len(set(int(i) for i in positions)),
                       "mean_reward": float(np.mean(rewards_flat)), "zero_advantage_groups": zero_groups,
                       "mean_execution_seconds": float(np.mean(execution_times)),
                       "mean_generation_seconds": float(np.mean(generation_times)),
                       "episode_seconds": time.perf_counter() - started,
                       **update, **cuda_memory(policy.device)}
            if hasattr(executor, "runtime"):
                metrics["execution_runtime"] = executor.runtime()
            append_json(run / "metrics.jsonl", metrics)
            print(metrics, flush=True)
            if (episode+1) % config.checkpoint_every == 0 or episode+1 == config.episodes:
                save_checkpoint(run / f"episode_{episode+1:04d}.pt", policy, optimizer,
                                config, episode+1, sampler, provenance)
            del groups
            yield {"run": run, "metrics": metrics}
        json_write(run / "status.json", {"status": "training_complete", "evaluation_completed": False,
                                          "completed_episodes": config.episodes})
    except Exception as exc:
        json_write(run / "status.json", {"status": "failed", "error": repr(exc),
                                          "evaluation_completed": False})
        raise
    finally:
        if policy.device.type == "cuda":
            torch.cuda.empty_cache()
    return run

def train(config, policy, examples, executor, provenance, resume=None):
    """One-shot wrapper over the same inspectable episode iterator."""
    iterator = train_episodes(config, policy, examples, executor, provenance, resume)
    while True:
        try:
            next(iterator)
        except StopIteration as completed:
            return completed.value

register_implementation("training-definition")


## 11. Complete four-way evaluation implementation

Static Predict, static CoT, initial untrained policy, and trained policy are evaluated sequentially on the same held-out examples with compatible settings. Numerical exact match and binary LLM fallback accuracy, latency components, failures and pipeline frequencies are saved separately.


In [12]:
"""Matched four-way held-out evaluation, with separate exact and semantic metrics."""
from collections import Counter
from pathlib import Path
import time
import torch


def evaluate(config, policy, examples, executor, checkpoint, provenance):
    if not examples or any(row.split != "test" for row in examples):
        raise ValueError("Evaluation requires official held-out test examples")
    run = new_run(config.output_root, "evaluation")
    payload = load_checkpoint(checkpoint, policy)
    initial = payload["provenance"]["initial_checkpoint"]
    training_config = payload["config"]
    initial_hash = artifact_hash(initial)
    expected_initial_hash = payload["provenance"].get("initial_checkpoint_sha256")
    if expected_initial_hash and initial_hash != expected_initial_hash:
        raise ValueError("Initial-policy checkpoint identity changed")
    # Scientific comparison settings cannot drift between train/untrained/trained.
    allow = {"output_root", "evaluation_limit", "device", "cpu_threads", "checkpoint_every"}
    for key, value in config.to_dict().items():
        if key not in allow and training_config[key] != value:
            raise ValueError(f"Evaluation config differs from checkpoint: {key}")
    if payload["provenance"].get("execution") != provenance.get("execution"):
        raise ValueError("Execution model identity/settings changed since training")
    json_write(run / "manifest.json", {**provenance, "config": config.to_dict(),
        "checkpoint": str(Path(checkpoint).resolve()), "checkpoint_sha256": artifact_hash(checkpoint),
        "completed_episodes": payload["completed_episodes"], "initial_checkpoint": initial,
        "initial_checkpoint_sha256": initial_hash, "metric": "normalized_numeric_exact_match",
        "semantic_metric": "binary_LLM_fallback", "judge_excluded_from_inference_latency": True})
    del payload  # Avoid retaining a second policy/optimizer copy during evaluation.
    summary = {}
    if policy.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for method in ("static_predict", "static_cot", "untrained_policy", "trained_policy"):
        if method == "untrained_policy":
            load_checkpoint(initial, policy)
        elif method == "trained_policy":
            load_checkpoint(checkpoint, policy)
        seed_everything(config.seed, config.cpu_threads)
        exact, semantic, errors, judge_errors = 0, 0, 0, 0
        generation_total, execution_total, judge_total = 0.0, 0.0, 0.0
        pipelines = Counter()
        for example in examples:
            started = time.perf_counter()
            if method.startswith("static_"):
                pipeline = ("Predict" if method == "static_predict" else "CoT", "question -> answer", "stop")
                generation_seconds = 0.0
            else:
                pipeline = policy.generate(example.question).pipeline
                generation_seconds = time.perf_counter() - started
            execution = executor.execute(example.question, pipeline)
            judged = time.perf_counter()
            result = compute_reward(example.question, execution.response, example.answer,
                                    executor.judge if config.semantic_fallback else None, binary=True)
            judge_seconds = time.perf_counter() - judged
            exact += int(result.exact)
            semantic += int(result.value == 1.0)
            errors += int(execution.error is not None)
            judge_errors += int(result.error is not None)
            generation_total += generation_seconds
            execution_total += execution.seconds
            judge_total += judge_seconds
            pipelines[" | ".join(pipeline)] += 1
            append_json(run / "predictions.jsonl", {"method": method, "example": example.to_dict(),
                "pipeline": pipeline, "execution": execution.to_dict(), "score": result.to_dict(),
                "generation_seconds": generation_seconds, "judge_seconds": judge_seconds})
            print(f"{method} test_index={example.index} exact={result.exact} semantic={result.value}", flush=True)
        n = len(examples)
        summary[method] = {"count": n, "exact_match_accuracy": exact/n,
                           "LLM_fallback_accuracy": semantic/n,
                           "mean_inference_seconds": (generation_total+execution_total)/n,
                           "mean_policy_generation_seconds": generation_total/n,
                           "mean_execution_seconds": execution_total/n, "mean_judge_seconds": judge_total/n,
                           "execution_failures": errors, "judge_parse_failures": judge_errors,
                           "pipeline_frequencies": dict(pipelines), **cuda_memory(policy.device)}
        if hasattr(executor, "runtime"):
            summary[method]["execution_runtime"] = executor.runtime()
        json_write(run / "results.json", summary)
    json_write(run / "status.json", {"status": "evaluation_complete", "examples_per_method": len(examples)})
    return run

register_implementation("evaluation-definition")


## 12. Offline tests against this notebook's own functions

These tests use tiny explicitly scripted fixtures, not GPT-2/Llama benchmark outputs.
They exercise numerical scoring, token collisions, advantages/loss/gradients,
accumulation equivalence, checkpoint/RNG resume, split guards and evaluator wiring.
No test imports `autodspy_reproduction`, and no LLM server is contacted.


In [13]:
"""Offline engineering tests; fixtures never represent Llama benchmark results."""
from dataclasses import replace
from decimal import Decimal
from pathlib import Path
from types import SimpleNamespace
import contextlib
import io
import json
import tempfile
import unittest
from unittest.mock import Mock, patch
import numpy as np
import torch


class Inputs(dict):
    def to(self, device):
        return Inputs({key: value.to(device) for key, value in self.items()})


class FixtureTokenizer:
    def encode(self, text, add_special_tokens=False):
        if text in MODULES:
            return [MODULES.index(text)]
        if text == "stop":
            return [9]
        return [2 + SIGNATURES.index(text) % 7, 11]

    def __call__(self, text, **kwargs):
        return Inputs(input_ids=torch.tensor([[0, 1]]))


class FixtureModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.logits = torch.nn.Parameter(torch.linspace(-0.2, 0.2, 12))

    def forward(self, **kwargs):
        return SimpleNamespace(logits=self.logits[None, None, :].expand(1, 2, -1))


class FixtureExecutor:
    def execute(self, prompt, pipeline):
        return Execution("[42]" if pipeline[0] == "CoT" else "[0]", 0.001)

    def judge(self, prompt):
        return "[0]"


def fixture_policy():
    return Policy(FixtureModel(), FixtureTokenizer())


class NumericalTests(unittest.TestCase):
    def test_signed_commas_decimals(self):
        cases = {"[-1,234.50]": Decimal("-1234.5"), "[+00012.00]": Decimal("12"),
                 "Result: -.5": Decimal("-.5"), "3 then 4.0": Decimal("4"),
                 "reasoning 10\n#### -2": Decimal("-2"), "[0]": Decimal("0")}
        for text, expected in cases.items():
            with self.subTest(text=text):
                self.assertEqual(normalize_number(text), expected)
        self.assertTrue(exact_match("[1,000.00]", "1000"))
        self.assertFalse(exact_match("[12]", "2"))
        self.assertIsNone(normalize_number("nothing"))
        self.assertIsNone(normalize_number("[3 or 4]"))

    def test_canonical_preserves_value(self):
        self.assertEqual(canonical_answer("2+2=4\n#### -1,000.50"), "-1,000.50")
        with self.assertRaises(ValueError):
            canonical_answer("42")

    def test_binary_judge_rejects_substring_false_positive(self):
        self.assertEqual(parse_judge_score("[1.0]", binary=True), 1)
        self.assertEqual(parse_judge_score("[0.75]"), .75)
        for text in ("[0.1]", "[10]", "nan", "score 1 or 0", "[inf]"):
            with self.assertRaises(ValueError):
                parse_judge_score(text, binary=True)

    def test_reward_exact_and_fallback(self):
        judge = lambda _: "[0.75]"
        self.assertEqual(compute_reward("q", "[42]", "42", judge).value, 1)
        result = compute_reward("q", "[0]", "42", judge)
        self.assertEqual(result.value, .75)
        self.assertTrue(result.fallback_used)
        self.assertEqual(compute_reward("q", None, "42", judge).value, 0)
        self.assertIsNotNone(compute_reward("q", "[0]", "42", judge, binary=True).error)

    def test_infrastructure_errors_propagate(self):
        def broken(_):
            raise ConnectionError("offline")
        with self.assertRaises(ConnectionError):
            compute_reward("q", "[0]", "42", broken)


class LossTests(unittest.TestCase):
    def test_centered_and_population_normalization(self):
        self.assertTrue(torch.allclose(group_advantages([0, 1, 1]), torch.tensor([-2/3, 1/3, 1/3])))
        standardized = group_advantages([0, 1, 1], "standardized")
        self.assertAlmostEqual(float(standardized.std(correction=0)), 1, places=6)
        for value in (0., 1., .1234567):
            self.assertTrue(torch.equal(group_advantages([value]*5), torch.zeros(5)))
            self.assertTrue(torch.equal(group_advantages([value]*5, "standardized"), torch.zeros(5)))
        with self.assertRaises(ValueError):
            group_advantages([0, float("nan")])

    def test_notebook_loss_matches_released_formula(self):
        current = torch.tensor([[-.2, -.7, 0], [-.9, -.3, 0]], requires_grad=True)
        old = current.detach().clone().requires_grad_()
        loss = group_loss(current, old, [1., 0.], torch.zeros_like(current))
        expected = -(current[0].sum()*.5 + current[1].sum()*-.5)
        self.assertTrue(torch.allclose(loss, expected))
        loss.backward()
        self.assertIsNone(old.grad)
        self.assertTrue(torch.isfinite(current.grad).all())
        self.assertGreater(float(current.grad.abs().sum()), 0)

    def test_clipping_both_advantage_signs(self):
        logs = torch.log(torch.tensor([1.5, .5], requires_grad=True))
        loss = action_loss(logs, torch.zeros(2), torch.tensor([1., -1.]), torch.zeros(2), "clipped")
        self.assertTrue(torch.allclose(loss, torch.tensor([-1.2, .8])))

    def test_clipped_entropy_and_behavior_detachment(self):
        current = torch.tensor(-.5, requires_grad=True)
        old = torch.tensor(-.5, requires_grad=True)
        entropy = torch.tensor(.7, requires_grad=True)
        loss = action_loss(current, old, 1., entropy, "clipped", .2, .01)
        self.assertAlmostEqual(float(loss.detach()), -1.007, places=6)
        loss.backward()
        self.assertIsNone(old.grad)
        self.assertAlmostEqual(float(current.grad), -1, places=6)
        self.assertAlmostEqual(float(entropy.grad), -.01, places=6)

    def test_equal_rewards_zero_gradient(self):
        logits = torch.tensor([[-.5, -.3], [-.7, -.2]], requires_grad=True)
        loss = group_loss(logits, logits.detach(), [1, 1], torch.zeros_like(logits))
        loss.backward()
        self.assertEqual(float(loss), 0)
        self.assertTrue(torch.equal(logits.grad, torch.zeros_like(logits)))


class PolicyTests(unittest.TestCase):
    def test_duplicate_tokens_are_separate_action_slots(self):
        tokenizer = FixtureTokenizer()
        logits = torch.arange(12, dtype=torch.float32, requires_grad=True)
        actions = (SIGNATURES[0], SIGNATURES[7], SIGNATURES[1])
        dist = action_distribution(logits, tokenizer, actions)
        expected = torch.softmax(logits[[2, 2, 3]], -1)
        self.assertTrue(torch.allclose(dist.probs, expected))
        self.assertEqual(float(dist.probs[0]), float(dist.probs[1]))
        dist.log_prob(torch.tensor(0)).backward()
        self.assertGreater(float(logits.grad.abs().sum()), 0)

    def test_generation_stop_and_replay_probability(self):
        policy = fixture_policy()
        trajectory = policy.generate("q")
        self.assertEqual(len(trajectory.pipeline), 3)
        self.assertEqual(trajectory.pipeline[-1], "stop")
        self.assertEqual(trajectory.decisions[-1].old_log_prob, 0)
        for decision in trajectory.decisions:
            new, _ = policy.score(decision)
            self.assertAlmostEqual(float(new.detach()), decision.old_log_prob, places=6)
        self.assertIsNotNone(pipeline_pair(trajectory.pipeline))
        self.assertIsNotNone(pipeline_pair(trajectory.pipeline[:-1]))
        self.assertIsNone(pipeline_pair(("ReAct", "question -> answer")))
        self.assertIsNone(pipeline_pair(("stop",)))

    def test_replay_updates_parameters(self):
        seed_everything(42)
        policy = fixture_policy()
        groups = [([policy.generate("q") for _ in range(5)], [0, 1, 0, 1, 0])]
        before = policy.model.logits.detach().clone()
        metrics = update_episode(policy, torch.optim.Adam(policy.model.parameters(), lr=1e-4), groups, Config())
        self.assertGreater(metrics["gradient_norm"], 0)
        self.assertFalse(torch.equal(before, policy.model.logits))

    def test_accumulated_replay_matches_full_graph_gradient(self):
        seed_everything(42)
        policy = fixture_policy()
        groups = [([policy.generate("q") for _ in range(5)], [0, 1, 0, 1, 0]),
                  ([policy.generate("q2") for _ in range(5)], [1, 0, 1, 0, 1])]
        expected_policy = fixture_policy()
        loss = 0
        for trajectories, rewards in groups:
            advantages = group_advantages(rewards)
            for trajectory, advantage in zip(trajectories, advantages):
                for decision in trajectory.decisions[:-1]:
                    logprob, _ = expected_policy.score(decision)
                    loss = loss-logprob*advantage/len(groups)
        loss.backward()
        expected_gradient = expected_policy.model.logits.grad.clone()
        # SGD with lr=1 makes its parameter delta exactly the accumulated gradient.
        before = policy.model.logits.detach().clone()
        update_episode(policy, torch.optim.SGD(policy.model.parameters(), lr=1), groups, Config())
        self.assertTrue(torch.allclose(before-policy.model.logits, expected_gradient, atol=1e-6))


class RunTests(unittest.TestCase):
    def test_dspy_execution_fields_and_adapter_failure(self):
        import dspy
        from dspy.utils.exceptions import AdapterParseError
        executor = DSPyExecutor(Config(device="cpu"))
        self.assertEqual(executor.lm.kwargs["num_gpu"], 0)
        self.assertEqual(executor.lm.kwargs["num_ctx"], 8192)
        self.assertFalse(executor.lm.cache)
        program = Mock(return_value=dspy.Prediction(summary="[42]"))
        with patch.object(dspy, "Predict", return_value=program):
            result = executor.execute("q", ("Predict", "text -> summary", "stop"))
        self.assertEqual(result.response, "[42]")
        self.assertIn("text", program.call_args.kwargs)
        self.assertIn("square brackets", program.call_args.kwargs["text"])
        failed = Mock(side_effect=AdapterParseError("test", dspy.Signature("text -> summary"), "malformed"))
        with patch.object(dspy, "Predict", return_value=failed):
            result = executor.execute("q", ("Predict", "text -> summary"))
        self.assertIsNone(result.response)
        self.assertIn("adapter_parse_error", result.error)

    def test_configs_and_dataset_split_guard(self):
        full = FULL_CONFIG
        smoke = SMOKE_CONFIG
        self.assertEqual((full.episodes, full.samples_per_episode, full.group_size), (200, 200, 5))
        self.assertEqual((smoke.episodes, smoke.group_size), (1, 5))
        with self.assertRaises(ValueError):
            replace(full, execution_model="llama3.2:3b").validate()
        check_no_leakage([Example(0, "a", "#### 1", "1", "train")],
                         [Example(0, "b", "#### 2", "2", "test")])
        with self.assertRaises(ValueError):
            check_no_leakage([Example(0, "a", "#### 1", "1", "train")],
                             [Example(0, "a", "#### 1", "1", "test")])

    def test_checkpoint_refuses_overwrite_and_restores_rng(self):
        seed_everything(42)
        policy = fixture_policy()
        sampler = np.random.default_rng(42)
        with tempfile.TemporaryDirectory() as directory:
            file = Path(directory)/"checkpoint.pt"
            save_checkpoint(file, policy, None, Config(), 0, sampler, {})
            expected = torch.rand(3)
            load_checkpoint(file, policy, sampler=sampler, restore_rng=True)
            self.assertTrue(torch.equal(torch.rand(3), expected))
            with self.assertRaises(FileExistsError):
                save_checkpoint(file, policy, None, Config(), 0, sampler, {})

    def test_training_checkpoint_resume_and_four_way_evaluation(self):
        examples = [Example(i, f"fixture {i}", "#### 42", "42", "train") for i in range(3)]
        with tempfile.TemporaryDirectory() as directory, contextlib.redirect_stdout(io.StringIO()):
            config = Config(device="cpu", episodes=2, samples_per_episode=2, checkpoint_every=1,
                            output_root=directory)
            provenance = {"data": {"kind": "fixture"}, "policy": {"kind": "fixture"},
                          "execution": {"kind": "fixture"}}
            seed_everything(42)
            policy = fixture_policy()
            run = train(config, policy, examples, FixtureExecutor(), provenance)
            final = policy.model.logits.detach().clone()
            checkpoint = run/"episode_0002.pt"
            self.assertTrue(checkpoint.exists())
            metrics = [json.loads(line) for line in (run/"metrics.jsonl").read_text().splitlines()]
            self.assertEqual(len(metrics), 2)
            self.assertEqual(metrics[0]["pipeline_evaluations"], 10)
            resumed_policy = fixture_policy()
            seed_everything(0)  # Must be replaced by checkpoint RNG.
            resumed = train(config, resumed_policy, examples, FixtureExecutor(), provenance,
                            resume=run/"episode_0001.pt")
            self.assertTrue(torch.equal(final, resumed_policy.model.logits))
            self.assertEqual(json.loads((resumed/"status.json").read_text())["completed_episodes"], 2)
            held_out = [Example(0, "held out fixture", "#### 42", "42", "test")]
            eval_run = evaluate(config, policy, held_out, FixtureExecutor(), checkpoint, provenance)
            results = json.loads((eval_run/"results.json").read_text())
            self.assertEqual(set(results), {"static_predict", "static_cot", "untrained_policy", "trained_policy"})
            self.assertTrue(all(row["count"] == 1 for row in results.values()))


In [14]:
suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromTestCase(test_class)
                           for test_class in (NumericalTests, LossTests, PolicyTests, RunTests))
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Fix offline validation before training."


test_binary_judge_rejects_substring_false_positive (__main__.NumericalTests.test_binary_judge_rejects_substring_false_positive) ... ok
test_canonical_preserves_value (__main__.NumericalTests.test_canonical_preserves_value) ... ok
test_infrastructure_errors_propagate (__main__.NumericalTests.test_infrastructure_errors_propagate) ... ok
test_reward_exact_and_fallback (__main__.NumericalTests.test_reward_exact_and_fallback) ... ok
test_signed_commas_decimals (__main__.NumericalTests.test_signed_commas_decimals) ... ok
test_centered_and_population_normalization (__main__.LossTests.test_centered_and_population_normalization) ... ok
test_clipped_entropy_and_behavior_detachment (__main__.LossTests.test_clipped_entropy_and_behavior_detachment) ... ok
test_clipping_both_advantage_signs (__main__.LossTests.test_clipping_both_advantage_signs) ... ok
test_equal_rewards_zero_gradient (__main__.LossTests.test_equal_rewards_zero_gradient) ... ok
test_notebook_loss_matches_released_formula (__main__.L

## 13. Verify the live execution model before GPU allocation

Start Ollama externally. This cell contacts only the configured endpoint and prints the verified version, model digest, quantization and generation settings. Connection failures must be resolved before proceeding.


In [15]:
config.validate()
seed_everything(config.seed, config.cpu_threads)
execution_manifest = preflight(config)
print(json.dumps({key: execution_manifest[key]
                  for key in ("version", "model", "endpoint", "settings")}, indent=2))


{
  "version": {
    "version": "0.33.2"
  },
  "model": {
    "name": "llama3.1:8b",
    "model": "llama3.1:8b",
    "modified_at": "2026-09-26T19:49:04.8691701+06:00",
    "size": 4920753328,
    "digest": "46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e",
    "details": {
      "parent_model": "",
      "format": "gguf",
      "family": "llama",
      "families": [
        "llama"
      ],
      "parameter_size": "8.0B",
      "quantization_level": "Q4_K_M",
      "context_length": 131072,
      "embedding_length": 4096
    },
    "capabilities": [
      "completion",
      "tools"
    ]
  },
  "endpoint": "http://localhost:11434",
  "settings": {
    "num_gpu": 0,
    "temperature": 0.0,
    "num_predict": 4096,
    "num_ctx": 8192,
    "seed": 42,
    "keep_alive": "5m",
    "timeout": 300,
    "cache": false
  }
}


## 14. Load the official training split

Default full pool: 7,473 examples. Each episode draws its own 200 indices with replacement; this is not a fixed 200-example training set. Original raw answers remain in every logged example.


In [16]:
train_examples, train_data_manifest = load_examples(config, "train")
print("Official train rows:", train_data_manifest["official_rows"])
print("Selected train pool:", len(train_examples))
print(train_examples[0].to_dict())


x:\Adaptive GRPO for AutoDSPy\Code\AUTODSPy\venv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Official train rows: 7473
Selected train pool: 7473
{'index': 0, 'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'raw_answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'answer': '72', 'split': 'train'}


## 15. Initialize pretrained GPT-2 and the DSPy executor

Fresh weights are loaded explicitly; existing checkpoints are not silently loaded. Rerunning this cell resets the policy, so restart/reinitialize the training iterator afterwards.


In [17]:
seed_everything(config.seed, config.cpu_threads)
policy = load_policy(config)
executor = DSPyExecutor(config)
policy_manifest = {
    "model": config.policy_model, "revision": config.policy_revision,
    "parameters": sum(parameter.numel() for parameter in policy.model.parameters()),
    "first_token_collisions": policy.token_collisions(), "dropout": 0.0,
    "precision": "float32", "max_tokens": config.max_policy_tokens,
}
training_provenance = {
    "environment": environment_manifest(), "data": train_data_manifest,
    "execution": execution_manifest, "policy": policy_manifest,
}
print(json.dumps(policy_manifest, indent=2))
print("Policy device:", policy.device)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1413.34it/s]


{
  "model": "gpt2",
  "revision": "607a30d783dfa663caf39e06633721c8d4cfcd7e",
  "parameters": 124439808,
  "first_token_collisions": {
    "25652": [
      "question -> answer",
      "question -> reasoning",
      "question -> hypothesis"
    ],
    "5239": [
      "text -> summary",
      "text -> response"
    ],
    "45573": [
      "problem -> solution",
      "problem_description -> explanation"
    ],
    "22866": [
      "context -> summary",
      "context -> briefing"
    ],
    "16963": [
      "prompt -> response",
      "prompt -> generated_text"
    ],
    "22766": [
      "query -> response",
      "query -> generated_text"
    ]
  },
  "dropout": 0.0,
  "precision": "float32",
  "max_tokens": 1024
}
Policy device: cuda:0


## 16. Optional live pipeline/reward probe

Set `RUN_EXECUTION_PROBE=True` to inspect a sampled pipeline, output, reward and latency for the first training question. This is an integration check, not a benchmark.


In [18]:
RUN_EXECUTION_PROBE = False
if RUN_EXECUTION_PROBE:
    example = train_examples[0]
    trajectory = policy.generate(example.question)
    execution = executor.execute(example.question, trajectory.pipeline)
    reward = compute_reward(example.question, execution.response, example.answer,
                            executor.judge if config.semantic_fallback else None)
    print(json.dumps({"example": example.to_dict(), "trajectory": trajectory.to_dict(),
                      "execution": execution.to_dict(), "reward": reward.to_dict()}, indent=2))
    assert execution.response is not None and not execution.error and not reward.error


## 17. Initialize interactive training or resume

Keep `RESUME_CHECKPOINT=None` for fresh training. To resume, set a real completed
checkpoint path here, after initializing a fresh policy with the same configuration.
The iterator refuses changed config, model/data/execution identity or implementation.
Notebook-created checkpoints resume through this notebook; CLI checkpoints resume
through the CLI because implementation provenance differs. Avoid rerunning this cell
while an iterator is active. For a new independent experiment, restart the kernel.


In [19]:
RESUME_CHECKPOINT = None  # Example: "runs/reproduction-notebook/RUN/episode_0010.pt"
if "training_iterator" in globals() and not globals().get("training_finished", False):
    raise RuntimeError("An iterator already exists. Continue it, or restart the kernel for a fresh/resumed run.")
seed_everything(config.seed, config.cpu_threads)  # Probe sampling must not affect training RNG.
training_iterator = train_episodes(config, policy, train_examples, executor,
                                   training_provenance, resume=RESUME_CHECKPOINT)
training_run = None
latest_metrics = None
training_finished = False

def advance_training():
    global training_run, latest_metrics, training_finished
    if training_finished:
        print("Training already completed:", training_run)
        return None
    try:
        result = next(training_iterator)
    except StopIteration as completed:
        training_run = completed.value
        training_finished = True
        print("Training complete; artifacts:", training_run.resolve())
        return None
    training_run, latest_metrics = result["run"], result["metrics"]
    # Finalize status/finally after the final yielded episode, without another user action.
    if latest_metrics["episode"] == config.episodes:
        try:
            next(training_iterator)
        except StopIteration as completed:
            training_run = completed.value
            training_finished = True
        else:
            raise RuntimeError("Unexpected episode beyond configured budget")
    print("Artifacts:", training_run.resolve())
    return latest_metrics

print("Prepared", MODE, "training; no episode has run yet.")


Prepared full training; no episode has run yet.


## 18. Run exactly one episode and inspect it

Set `RUN_ONE_EPISODE=True`. Each execution advances exactly one episode of the selected configuration. Full mode still uses 200 prompts and K=5. In smoke mode this finishes the whole one-episode experiment. Afterwards inspect `latest_metrics` or the artifact cell below.


In [ ]:
RUN_ONE_EPISODE = True
if RUN_ONE_EPISODE:
    episode_metrics = advance_training()
    if episode_metrics is not None:
        print(json.dumps(episode_metrics, indent=2))


episode=1 sample=1/200 train_index=666 rewards=[1.0, 1.0, 1.0, 1.0, 1.0]
episode=1 sample=2/200 train_index=5783 rewards=[0.0, 1.0, 0.0, 0.0, 1.0]
episode=1 sample=3/200 train_index=4891 rewards=[1.0, 1.0, 1.0, 1.0, 1.0]
episode=1 sample=4/200 train_index=3279 rewards=[0.0, 1.0, 0.0, 0.0, 0.0]
episode=1 sample=5/200 train_index=3235 rewards=[1.0, 1.0, 1.0, 1.0, 1.0]


## 19. Run all remaining episodes

Set `RUN_REMAINING_TRAINING=True` and execute this cell for full training. It continues the existing iterator after any manually inspected episodes. Every prompt group's rewards and each episode's loss/gradient/memory are printed; trajectories and metrics stream to disk. No reduced workload is substituted for full mode.


In [ ]:
RUN_REMAINING_TRAINING = False
if RUN_REMAINING_TRAINING:
    while not training_finished:
        advance_training()
else:
    print("Training disabled. Set RUN_REMAINING_TRAINING=True to start/continue", MODE, "training.")


## 20. Inspect real training artifacts and GPU memory

In another terminal, `nvidia-smi --query-gpu=timestamp,name,memory.used,memory.total --format=csv -l 2` monitors total-device memory. Logged peaks here cover only PyTorch policy allocations. Loss/reward values below are read from actual files; no example results are invented.


In [ ]:
if training_run is not None:
    print("Run:", training_run.resolve())
    checkpoints = sorted(training_run.glob("*.pt"))
    print("Checkpoints:", [file.name for file in checkpoints])
    status_file = training_run / "status.json"
    if status_file.exists():
        print("Status:", json.loads(status_file.read_text()))
    metrics_file = training_run / "metrics.jsonl"
    if metrics_file.exists():
        training_metrics = [json.loads(line) for line in metrics_file.read_text().splitlines()]
        print("Latest metrics:", json.dumps(training_metrics[-1], indent=2))
    print("CUDA memory:", cuda_memory(policy.device))
else:
    print("No episode executed in this session yet.")


## 21. Run compatible four-way held-out evaluation

Enable `RUN_EVALUATION` only after training. `EVALUATION_CHECKPOINT=None` automatically
uses this session's final checkpoint; set an explicit path for an earlier notebook/CLI
run with matching config and execution settings. To evaluate an earlier smoke run,
use `MODE="smoke"` and initialize its matching settings. This cell loads the official
test split, checks overlap, and executes all four methods. Smoke tests two cases;
full tests all 1,319 cases and requires substantial execution time. Evaluation leaves
the policy at trained weights and resets RNG per method; restart/resume from a saved
checkpoint if further training is needed, rather than evaluating mid-iterator.


In [ ]:
RUN_EVALUATION = False
EVALUATION_CHECKPOINT = None  # Or an explicit isolated-reproduction .pt path.
evaluation_run = None
if RUN_EVALUATION:
    if not globals().get("training_finished", False) and globals().get("training_run") is not None:
        raise RuntimeError("Finish the active training iterator before evaluation.")
    if EVALUATION_CHECKPOINT is not None:
        checkpoint = Path(EVALUATION_CHECKPOINT)
    else:
        if not globals().get("training_finished", False) or globals().get("training_run") is None:
            raise RuntimeError("Complete training first, or set EVALUATION_CHECKPOINT explicitly.")
        checkpoint = training_run / f"episode_{config.episodes:04d}.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    test_examples, test_data_manifest = load_examples(config, "test")
    check_no_leakage(train_examples, test_examples)
    evaluation_provenance = {
        "environment": environment_manifest(), "data": test_data_manifest,
        "execution": execution_manifest, "policy": policy_manifest,
    }
    evaluation_run = evaluate(config, policy, test_examples, executor, checkpoint, evaluation_provenance)
    print("Evaluation artifacts:", evaluation_run.resolve())


## 22. Inspect the actual comparison results

These are engineering/smoke results until full training and compatible held-out evaluation finish. Unresolved published protocol ambiguities still limit an exact claim of replication.


In [ ]:
if evaluation_run is not None:
    results = json.loads((evaluation_run / "results.json").read_text())
    for method, measured in results.items():
        print(method)
        print(json.dumps(measured, indent=2))
else:
    print("No held-out evaluation was run in this session.")
